# Subgen — Subdomain Generation Model

Full pipeline: preprocess → train tokenizer → pack dataset → train → generate

**Requirements:** Upload `domains_export.csv` to Google Drive or directly to Colab.

Data pipeline artifacts are stored on Google Drive for persistence. Training data
and checkpoints are copied to local SSD for speed, then synced back to Drive.

## 0. Setup

In [ ]:
# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

import os
import shutil

# Drive directory — persists across sessions
DRIVE_DIR = '/content/drive/MyDrive/subgen'
os.makedirs(DRIVE_DIR, exist_ok=True)

# Local SSD directory — fast I/O for training
LOCAL_DIR = '/content/subgen'
os.makedirs(LOCAL_DIR, exist_ok=True)

print(f'Drive dir (persistent): {DRIVE_DIR}')
print(f'Local dir (fast I/O):   {LOCAL_DIR}')

In [ ]:
# Install dependencies (flash-attn for fast training + inference)
!pip install -q torch transformers tokenizers tldextract wandb numpy tqdm accelerate
!pip install -q flash-attn --no-build-isolation 2>/dev/null || echo 'flash-attn not available, will fall back to SDPA (slower)'

In [ ]:
# Check GPU
import torch
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {gpu} ({vram:.1f} GB)')
    print(f'BF16 support: {torch.cuda.is_bf16_supported()}')
else:
    print('WARNING: No GPU detected. Training will be very slow.')

In [ ]:
# Clone or update repo on local SSD
REPO_DIR = os.path.join(LOCAL_DIR, 'repo')

if os.path.isdir(REPO_DIR):
    print('Repo already exists, pulling latest...')
    !cd {REPO_DIR} && git pull
else:
    !git clone https://github.com/c3l3si4n/subgen.git {REPO_DIR}

os.chdir(REPO_DIR)
print(f'Repo dir: {os.getcwd()}')
!ls

In [ ]:
import sys
sys.path.insert(0, os.path.join(REPO_DIR, 'src'))

## 1. Preprocess

In [ ]:
# Verify dataset exists — check local, Drive, then prompt upload
DATASET_PATH = os.path.join(REPO_DIR, 'dataset', 'domains_export.csv')
DRIVE_DATASET_LOCATIONS = [
    os.path.join(DRIVE_DIR, 'domains_export.csv'),
    os.path.join(DRIVE_DIR, 'repo', 'dataset', 'domains_export.csv'),
    '/content/drive/MyDrive/repo/dataset/domains_export.csv',
]

if not os.path.isfile(DATASET_PATH):
    os.makedirs(os.path.join(REPO_DIR, 'dataset'), exist_ok=True)
    found = None
    for loc in DRIVE_DATASET_LOCATIONS:
        if os.path.isfile(loc):
            found = loc
            break
    if found:
        print(f'Copying dataset from {found}...')
        shutil.copy2(found, DATASET_PATH)
    else:
        from google.colab import files
        print('Dataset not found. Upload domains_export.csv:')
        uploaded = files.upload()
        for name, data in uploaded.items():
            with open(DATASET_PATH, 'wb') as f:
                f.write(data)
            break

!wc -l {DATASET_PATH} | head -1

In [ ]:
# Set FORCE_REPROCESS = True when retraining from scratch with new data.
# This wipes ALL cached artifacts (data, tokenizer, checkpoints) on Drive and local SSD.
FORCE_REPROCESS = True

DATA_DIR = os.path.join(REPO_DIR, 'data')
DRIVE_DATA_DIR = os.path.join(DRIVE_DIR, 'data')

if FORCE_REPROCESS:
    print('FORCE_REPROCESS: wiping all cached artifacts...')
    # Wipe Drive caches
    for d in [DRIVE_DATA_DIR, os.path.join(DRIVE_DIR, 'tokenizer'), os.path.join(DRIVE_DIR, 'checkpoints')]:
        if os.path.isdir(d):
            shutil.rmtree(d)
            print(f'  Removed {d}')
    # Wipe local SSD caches
    for d in [DATA_DIR, os.path.join(LOCAL_DIR, 'checkpoints')]:
        if os.path.isdir(d):
            shutil.rmtree(d)
            print(f'  Removed {d}')

if os.path.isfile(os.path.join(DRIVE_DATA_DIR, 'train_sequences.txt')):
    print('Restoring preprocessed data from Drive...')
    os.makedirs(DATA_DIR, exist_ok=True)
    shutil.copytree(DRIVE_DATA_DIR, DATA_DIR, dirs_exist_ok=True)
    print('Done')
else:
    from data.preprocess import preprocess
    preprocess(
        input_path=DATASET_PATH,
        output_dir=DATA_DIR,
        max_subs_per_domain=500,
        sort_tmp_dir=LOCAL_DIR,  # use local SSD for sort temp files
    )
    # Save to Drive
    os.makedirs(DRIVE_DATA_DIR, exist_ok=True)
    for f in ['train_sequences.txt', 'val_sequences.txt']:
        src = os.path.join(DATA_DIR, f)
        if os.path.isfile(src):
            shutil.copy2(src, os.path.join(DRIVE_DATA_DIR, f))

In [ ]:
# Verify: check a few lines
!head -3 data/train_sequences.txt

## 2. Train Tokenizer

In [ ]:
TOKENIZER_DIR = os.path.join(REPO_DIR, 'tokenizer')
DRIVE_TOKENIZER_DIR = os.path.join(DRIVE_DIR, 'tokenizer')

if os.path.isfile(os.path.join(DRIVE_TOKENIZER_DIR, 'tokenizer.json')):
    print('Restoring tokenizer from Drive...')
    shutil.copytree(DRIVE_TOKENIZER_DIR, TOKENIZER_DIR, dirs_exist_ok=True)
else:
    from data.tokenizer_train import train_tokenizer
    train_tokenizer(
        input_files=[os.path.join(DATA_DIR, 'train_sequences.txt')],
        output_dir=TOKENIZER_DIR,
        # vocab_size default (4096) is set in tokenizer_train.py
    )
    # Save to Drive
    shutil.copytree(TOKENIZER_DIR, DRIVE_TOKENIZER_DIR, dirs_exist_ok=True)

## 3. Pack Dataset

In [ ]:
from transformers import PreTrainedTokenizerFast

tokenizer = PreTrainedTokenizerFast.from_pretrained(TOKENIZER_DIR)

if os.path.isfile(os.path.join(DRIVE_DATA_DIR, 'train.bin')):
    print('Restoring packed bins from Drive...')
    for f in ['train.bin', 'val.bin']:
        src = os.path.join(DRIVE_DATA_DIR, f)
        if os.path.isfile(src):
            shutil.copy2(src, os.path.join(DATA_DIR, f))
    print('Done')
else:
    from data.dataset import pretokenize_packed
    for split in ['train', 'val']:
        input_path = os.path.join(DATA_DIR, f'{split}_sequences.txt')
        output_path = os.path.join(DATA_DIR, f'{split}.bin')
        if os.path.exists(input_path):
            pretokenize_packed(input_path, output_path, tokenizer)
            # Save to Drive
            shutil.copy2(output_path, os.path.join(DRIVE_DATA_DIR, f'{split}.bin'))
        else:
            print(f'Skipping {split}: not found')

## 4. Train

In [ ]:
# Optional: set up W&B logging
USE_WANDB = False  # Set True and run `wandb login` if you want logging

if USE_WANDB:
    import wandb
    wandb.login()

In [ ]:
from data.dataset import DomainDataset, PackedDataCollator
from model.config import build_config, build_model
from transformers import PreTrainedTokenizerFast, Trainer, TrainingArguments

tokenizer = PreTrainedTokenizerFast.from_pretrained(TOKENIZER_DIR)

# Architecture variant: "default" (12×1024) or "wide" (8×1280, same param budget)
VARIANT = "default"

# Use FA2 if flash-attn is installed (3-5x faster than SDPA with 4D masks)
try:
    import flash_attn  # noqa: F401
    ATTN_IMPL = "flash_attention_2"
    print("Using FlashAttention 2 (optimized CUDA kernels)")
except ImportError:
    ATTN_IMPL = "sdpa"
    print("flash-attn not installed, falling back to SDPA (slower)")

config = build_config(
    vocab_size=len(tokenizer),
    variant=VARIANT,
    attn_implementation=ATTN_IMPL,
)
model = build_model(config)

# Load datasets from local SSD
train_dataset = DomainDataset(os.path.join(DATA_DIR, 'train.bin'))
val_dataset = DomainDataset(os.path.join(DATA_DIR, 'val.bin'))
print(f'Train rows: {len(train_dataset)}')
print(f'Val rows: {len(val_dataset)}')

In [ ]:
# Detect hardware capabilities
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
use_fp16 = torch.cuda.is_available() and not use_bf16

# Scale batch size to available VRAM
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
batch_size = 64 if vram_gb >= 35 else 32 if vram_gb >= 20 else 16 if vram_gb >= 14 else 8

# Target ~0.5M tokens per optimizer step for stable gradients.
# tokens_per_step = batch_size × grad_accum × seq_len
SEQ_LEN = 1024
TARGET_TOKENS = 524_288  # 0.5M tokens
grad_accum = max(1, TARGET_TOKENS // (batch_size * SEQ_LEN))

tokens_per_step = batch_size * grad_accum * SEQ_LEN
print(f'VRAM: {vram_gb:.1f} GB → batch_size={batch_size}, grad_accum={grad_accum}')
print(f'Tokens per step: {tokens_per_step:,} ({tokens_per_step / 1e6:.2f}M)')
print(f'Precision: {"bf16" if use_bf16 else "fp16" if use_fp16 else "fp32"}')

In [ ]:
# Checkpoints on local SSD for speed
OUTPUT_DIR = os.path.join(LOCAL_DIR, 'checkpoints')
DRIVE_CKPT_DIR = os.path.join(DRIVE_DIR, 'checkpoints')

# Restore checkpoint from Drive if resuming a session
if os.path.isdir(DRIVE_CKPT_DIR) and not os.path.isdir(OUTPUT_DIR):
    print('Restoring checkpoints from Drive...')
    shutil.copytree(DRIVE_CKPT_DIR, OUTPUT_DIR)
    print('Done')

# Work around PyTorch Inductor TF32 API conflict (pytorch/pytorch#166387)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# With a much larger dataset (100M+ subdomains), 1 epoch sees enough unique data.
# Bump to 2-3 if you want the model to revisit examples.
NUM_EPOCHS = 3

# Scale warmup to dataset size — ~2% of total steps is a good rule of thumb
train_rows = len(train_dataset)
steps_per_epoch = train_rows // (batch_size * grad_accum)
total_steps = steps_per_epoch * NUM_EPOCHS
warmup_steps = min(2000, int(total_steps * 0.02))

print(f'Dataset: {train_rows} rows')
print(f'Steps/epoch: {steps_per_epoch}, total steps: {total_steps}')
print(f'Warmup: {warmup_steps} steps')

# FA2 is memory-efficient (O(N) not O(N²)) — no need for gradient checkpointing
use_grad_ckpt = ATTN_IMPL != "flash_attention_2"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    gradient_accumulation_steps=grad_accum,
    learning_rate=6e-4,
    warmup_steps=warmup_steps,
    weight_decay=0.1,
    max_grad_norm=1.0,
    adam_beta1=0.9,
    adam_beta2=0.95,
    lr_scheduler_type='cosine',
    bf16=use_bf16,
    fp16=use_fp16,
    logging_steps=100,
    eval_strategy='steps',
    eval_steps=5000,
    save_steps=5000,
    save_total_limit=2,
    report_to='wandb' if USE_WANDB else 'none',
    run_name='subgen-150m-colab',
    optim='adamw_torch_fused',
    dataloader_num_workers=8,
    dataloader_persistent_workers=True,
    dataloader_prefetch_factor=4,
    dataloader_pin_memory=True,
    remove_unused_columns=False,
    gradient_checkpointing=use_grad_ckpt,
    torch_compile=True,
    label_smoothing_factor=0.05,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=PackedDataCollator(use_fa2=ATTN_IMPL == "flash_attention_2"),
    processing_class=tokenizer,
)

In [ ]:
# Auto-resume from checkpoint if available
resume_from = None
if os.path.isdir(OUTPUT_DIR):
    checkpoints = [d for d in os.listdir(OUTPUT_DIR) if d.startswith('checkpoint-')]
    if checkpoints:
        latest = max(checkpoints, key=lambda x: int(x.split('-')[1]))
        resume_from = os.path.join(OUTPUT_DIR, latest)
        print(f'Resuming from {resume_from}')

trainer.train(resume_from_checkpoint=resume_from)

In [ ]:
# Save final model locally and sync to Drive
final_dir = os.path.join(OUTPUT_DIR, 'final')
trainer.save_model(final_dir)
tokenizer.save_pretrained(final_dir)
print(f'Final model saved to {final_dir}')

# Sync checkpoints to Drive for persistence
print('Syncing checkpoints to Drive...')
if os.path.isdir(DRIVE_CKPT_DIR):
    shutil.rmtree(DRIVE_CKPT_DIR)
shutil.copytree(OUTPUT_DIR, DRIVE_CKPT_DIR)
print('Done')

## 5. Generate

In [ ]:
from generate import generate_subdomains, load_model

model, tokenizer = load_model(final_dir, device='auto')

candidates = generate_subdomains(
    model, tokenizer,
    root_domain='example.com',
    num_samples=20,
    temperature=0.9,
    min_p=0.05,
    repetition_penalty=1.3,
    temperature_sweep=True,
)

print(f'Generated {len(candidates)} candidates:')
for c in candidates:
    print(c)

## 6. Evaluate (held-out recall)

In [ ]:
from eval import evaluate_recall, load_val_domains

val_path = os.path.join(DATA_DIR, 'val_sequences.txt')
val_domains = load_val_domains(val_path, min_subs=5)
print(f'Found {len(val_domains)} domains with >= 5 subdomains')

results = evaluate_recall(
    model, tokenizer, val_domains,
    num_samples=5,
    batch_size=16,
    max_domains=50,  # quick eval; remove cap for full run
)

print(f'\nMacro recall: {results["macro_recall"]:.4f}')
print(f'Micro recall: {results["micro_recall"]:.4f}')

## 7. GRPO RL Fine-Tuning (Phase 2)

Uses [verl](https://github.com/verl-project/verl) with a rule-based reward signal to optimize
the SFT checkpoint. GRPO needs no critic model — memory-cheap, fits easily on a single GPU.

**Note:** DNS LogitsProcessor is intentionally NOT used during RL rollouts.
Invalid tokens are generated and penalized by the reward signal, giving the policy
gradient a cleaner learning signal. Keep the processor for final inference only.

In [ ]:
# Install verl and build RL dataset
!pip install -q verl pyarrow

from rl_dataset import build_rl_dataset

RL_PARQUET = os.path.join(DATA_DIR, 'rl_train.parquet')
build_rl_dataset(
    input_path=os.path.join(DATA_DIR, 'train_sequences.txt'),
    output_path=RL_PARQUET,
    tokenizer_dir=TOKENIZER_DIR,
    min_subs=5,
    prompt_ratio=0.8,
)

import pyarrow.parquet as pq
table = pq.read_table(RL_PARQUET)
print(f'RL dataset: {len(table)} examples')
print(table.schema)
print(table.slice(0, 3).to_pandas())

In [ ]:
# GRPO training with verl
# Uses the SFT checkpoint as the starting policy
import json
from reward import compute_score

# verl expects a reward function file — write a wrapper that verl can import
reward_wrapper = os.path.join(REPO_DIR, 'src', 'verl_reward.py')
with open(reward_wrapper, 'w') as f:
    f.write('''"""verl-compatible reward wrapper for subdomain generation."""
import json
import sys, os
sys.path.insert(0, os.path.dirname(__file__))
from reward import compute_score

def reward_function(solution_str, ground_truth_str, extra_info=None):
    """verl-compatible interface: ground_truth is JSON-encoded list."""
    gt = json.loads(ground_truth_str) if isinstance(ground_truth_str, str) else ground_truth_str
    return compute_score(solution_str, gt, extra_info)
''')

# GRPO config
GRPO_CONFIG = {
    "algorithm": "grpo",
    "model_path": final_dir,
    "tokenizer_path": TOKENIZER_DIR,
    "train_data_path": RL_PARQUET,
    "reward_function": reward_wrapper,
    "grpo": {
        "n": 16,              # samples per prompt
        "kl_coef": 0.001,     # low KL to allow policy to move
        "temperature": 0.9,
    },
    "training": {
        "learning_rate": 1e-5,
        "num_epochs": 1,
        "batch_size": 4,
        "gradient_accumulation_steps": 4,
        "max_new_tokens": 512,
        "bf16": True,
    },
    "output_dir": os.path.join(OUTPUT_DIR, "grpo"),
}

print("GRPO config:")
print(json.dumps(GRPO_CONFIG, indent=2))
print(f"\nTo run: use verl's GRPO trainer with this config.")
print(f"After RL, compare eval recall before/after with Section 6.")